# Hybrid Quantum-Classical Geospatial Classification

This notebook runs the hybrid quantum-classical experiments on the EuroSAT dataset.

## Setup
First, we clone the repository (if running from Colab) and install dependencies.

In [ ]:
# !git clone https://github.com/your-username/your-repo.git
# %cd your-repo

!pip install torchgeo pennylane lightning transformers timm rasterio segmentation-models-pytorch

## Imports

In [ ]:
import sys
import os
sys.path.append('.') # Ensure local modules are found

import torch
from src.data.dataset import get_dataset
from src.data.splitter import EuroSATSplitter
from src.models.backbones import BackboneFactory
from src.models.hybrid_model import HybridGeoModel
from run_experiments import run_experiment, CONFIGS

print("Imports successful.")

## Configuration
Select the experiment configuration to run.

In [ ]:
# Available configs:
print(list(CONFIGS.keys()))

selected_config = 'hybrid_resnet_angle_vqc' # @param ['baseline_resnet50', 'baseline_vit', 'hybrid_resnet_angle_vqc', 'hybrid_resnet_amplitude_vqc', 'hybrid_resnet_iqp_vqc', 'hybrid_vit_iqp_qaoa', 'hybrid_vit_qlstm']
n_qubits = 4 # @param {type:"slider", min:4, max:10, step:1}
epochs = 5 # @param {type:"integer"}
batch_size = 32 # @param {type:"integer"}
bands = 'RGB' # @param ['RGB', 'ALL']

class Args:
    def __init__(self):
        self.config = selected_config
        self.data_root = './data/EuroSAT'
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = 1e-4
        self.n_qubits = n_qubits
        self.bands = bands

args = Args()

## Run Experiment

In [ ]:
os.makedirs(args.data_root, exist_ok=True)
results = run_experiment(args.config, args)

## Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(results['loss'], label='Training Loss')
plt.title(f'Convergence: {args.config}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

print(f"Final Validation Accuracy: {results['val_acc'][-1]:.4f}")